# 02 — CNN Baseline (Colab + Drive)

Multi-label genre CNN on Drive mels. **Runtime → GPU**.

Needs: notebook 00 + 01 (`song_manifest.csv`).

Saves: `checkpoints/baseline/best.pt` and `results/02_baseline_test.json`.


## Colab + Drive (every notebook)

1. Open in **Google Colab**.
2. Run **Mount Drive** and click **Allow**.
3. Shared folder: `/content/drive/MyDrive/MTG_Instrument`
4. GPU **On** only for 02, 03, 07. Off for 00, 01, 04–06, 09.
5. Do **not** re-download mels after notebook 00.


In [ ]:
!pip install -q scikit-learn tqdm


## Mount Drive


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/MTG_Instrument")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for sub in ["dataset/logmel_songs", "annotations", "features", "checkpoints", "results"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["MTG_ROOT"] = str(DRIVE_ROOT)
print("Drive ready:", DRIVE_ROOT)


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, urllib.request
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get("MTG_ROOT", "/content/drive/MyDrive/MTG_Instrument"))
ROOT = DRIVE_ROOT
MEL_DIR = ROOT / "dataset" / "logmel_songs"
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host="github.com", port=443, timeout=5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    m = re.search(r"(\d+)", str(raw))
    return f"{int(m.group(1)):07d}" if m else None


def ensure_annotations():
    dest_train = ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if dest_train.exists():
        return
    if not check_internet():
        raise FileNotFoundError("Split TSVs missing and no Internet. Enable Internet and re-run.")
    print("Downloading annotation TSVs to Drive...")
    for rel in NEEDED_ANN:
        dest = ANN_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_ANN}/{rel}", dest)
        print(" ", dest)


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    """First column only — extra tag tabs break pandas read_csv."""
    name = f"autotagging_{subset}-{split}.tsv"
    path = ANN_DIR / "splits" / "split-0" / name
    if not path.exists():
        path = ANN_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    ids = set()
    with open(path, encoding="utf-8", errors="replace") as f:
        f.readline()
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            tid = normalize_track_id(line.split("\t")[0])
            if tid:
                ids.add(tid)
    print(f"{split:12s} {len(ids):6d} ids ← {path}")
    return ids


def iter_tsv_rows(path: Path):
    """Yield dict with TRACK_ID and remaining fields joined as TAGS."""
    with open(path, encoding="utf-8", errors="replace") as f:
        header = f.readline().strip().split("\t")
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if not parts:
                continue
            row = {"TRACK_ID": parts[0]}
            if len(parts) >= 6:
                row["TAGS"] = "\t".join(parts[5:])
            elif len(parts) > 1:
                row["TAGS"] = parts[-1]
            else:
                row["TAGS"] = ""
            yield row


ensure_annotations()
print("ROOT   ", ROOT)
print("MEL_DIR", MEL_DIR, "npy=", len(list(MEL_DIR.rglob("*.npy"))))
print("ANN_DIR", ANN_DIR)
print("MANIFEST", MANIFEST, "exists=", MANIFEST.exists())


## Load manifest + genre labels


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if not MANIFEST.exists():
    raise FileNotFoundError("Run 01 first — missing song_manifest.csv on Drive")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)

def load_genre_multihot(song_ids):
    paths = [ANN_DIR / "autotagging_genre.tsv", *ANN_DIR.rglob("*genre*.tsv")]
    tag_to_idx, rows = {}, {s: set() for s in song_ids}
    for path in paths:
        if not Path(path).exists():
            continue
        for rec in iter_tsv_rows(Path(path)):
            sid = normalize_track_id(rec["TRACK_ID"])
            if sid not in rows:
                continue
            for tag in rec.get("TAGS", "").replace("|", "\t").split("\t"):
                leaf = tag.strip().split("/")[-1].split("---")[-1]
                if not leaf or leaf.lower() in {"nan", "none", "tags", ""}:
                    continue
                tag_to_idx.setdefault(leaf, len(tag_to_idx))
                rows[sid].add(leaf)
        if tag_to_idx:
            print("tags from", path, len(tag_to_idx))
            break
    names = [None] * len(tag_to_idx)
    for t, i in tag_to_idx.items():
        names[i] = t
    Y = np.zeros((len(song_ids), len(names)), np.float32)
    for i, sid in enumerate(song_ids):
        for t in rows[sid]:
            Y[i, tag_to_idx[t]] = 1.0
    return Y, names

song_ids = manifest["song_id"].astype(str).tolist()
Y, TAG_NAMES = load_genre_multihot(song_ids)
print("Y", Y.shape, "pos", float(Y.mean()))
(RESULTS_DIR / "genre_tags.json").write_text(json.dumps(TAG_NAMES, indent=2))


## Dataset / model / train (best val PR-AUC checkpoint)


In [ ]:
class MelGenreDataset(Dataset):
    def __init__(self, df, Y, id_to_idx, max_windows=12, n_mels=96, n_frames=1366):
        self.df = df.reset_index(drop=True)
        self.Y, self.id_to_idx = Y, id_to_idx
        self.max_windows, self.n_mels, self.n_frames = max_windows, n_mels, n_frames

    def __len__(self):
        return len(self.df)

    def _fix2d(self, x):
        """Every song must become (n_mels, n_frames) or the batch cannot stack."""
        if x.shape[0] > self.n_mels:
            x = x[: self.n_mels]
        elif x.shape[0] < self.n_mels:
            x = np.pad(x, ((0, self.n_mels - x.shape[0]), (0, 0)))
        if x.shape[1] > self.n_frames:
            x = x[:, : self.n_frames]
        elif x.shape[1] < self.n_frames:
            x = np.pad(x, ((0, 0), (0, self.n_frames - x.shape[1])))
        return x

    def __getitem__(self, i):
        row = self.df.iloc[i]
        x = np.load(row["mel_abs"])
        if x.ndim == 2:
            x = x[None, ...]
        W = x.shape[0]
        x = x[: self.max_windows] if W >= self.max_windows else np.concatenate(
            [x, np.zeros((self.max_windows - W, *x.shape[1:]), x.dtype)], 0
        )
        x = self._fix2d(x.mean(0))  # (n_mels, n_frames)
        y = self.Y[self.id_to_idx[str(row["song_id"])]]
        return torch.tensor(x[None], dtype=torch.float32), torch.tensor(y)


id_to_idx = {s: i for i, s in enumerate(song_ids)}

def make_loader(split, bs=16, shuffle=False):
    sub = manifest[manifest["split"] == split]
    assert set(sub["split"].unique()) == {split}
    # num_workers=0 is required on Colab; workers + Drive .npy often crash
    return DataLoader(MelGenreDataset(sub, Y, id_to_idx), batch_size=bs, shuffle=shuffle, num_workers=0)

class BaselineCNN(nn.Module):
    def __init__(self, n_tags):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_tags))
    def forward(self, x):
        return self.head(self.features(x))

model = BaselineCNN(Y.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss()

def nan_safe(y_true, y_prob, kind="roc"):
    scores = []
    for k in range(y_true.shape[1]):
        if y_true[:, k].sum() in (0, len(y_true)):
            continue
        try:
            scores.append(roc_auc_score(y_true[:, k], y_prob[:, k]) if kind == "roc" else average_precision_score(y_true[:, k], y_prob[:, k]))
        except ValueError:
            continue
    return float(np.mean(scores)) if scores else float("nan")

@torch.no_grad()
def evaluate(loader):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        ps.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()); ys.append(y.numpy())
    yt, yp = np.concatenate(ys), np.concatenate(ps)
    return {"macro_roc_auc": nan_safe(yt, yp, "roc"), "macro_pr_auc": nan_safe(yt, yp, "pr")}

train_loader, val_loader, test_loader = make_loader("train", shuffle=True), make_loader("validation"), make_loader("test")
best_macro_map = 0.0
ckpt = CKPT_DIR / "baseline"; ckpt.mkdir(parents=True, exist_ok=True)
hist = []
for epoch in range(1, 11):
    model.train(); total = 0
    for x, y in tqdm(train_loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
        total += loss.item() * len(x)
    vm = evaluate(val_loader)
    hist.append({"epoch": epoch, "loss": total/len(train_loader.dataset), **vm})
    print(epoch, hist[-1])
    if vm["macro_pr_auc"] > best_macro_map:
        best_macro_map = vm["macro_pr_auc"]
        torch.save({"model": model.state_dict(), "tags": TAG_NAMES, "best_macro_map": best_macro_map, "epoch": epoch}, ckpt/"best.pt")
        print("  ✓ saved", best_macro_map)

state = torch.load(ckpt/"best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model"])
test_m = evaluate(test_loader)
print("TEST split-0", test_m)
pd.DataFrame(hist).to_csv(RESULTS_DIR/"02_baseline_history.csv", index=False)
(RESULTS_DIR/"02_baseline_test.json").write_text(json.dumps(test_m, indent=2))

class BaselineCNN(nn.Module):
    def __init__(self, n_tags):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_tags))
    def forward(self, x):
        return self.head(self.features(x))

model = BaselineCNN(Y.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss()

def nan_safe(y_true, y_prob, kind="roc"):
    scores = []
    for k in range(y_true.shape[1]):
        if y_true[:, k].sum() in (0, len(y_true)):
            continue
        try:
            scores.append(roc_auc_score(y_true[:, k], y_prob[:, k]) if kind == "roc" else average_precision_score(y_true[:, k], y_prob[:, k]))
        except ValueError:
            continue
    return float(np.mean(scores)) if scores else float("nan")

@torch.no_grad()
def evaluate(loader):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        ps.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()); ys.append(y.numpy())
    yt, yp = np.concatenate(ys), np.concatenate(ps)
    return {"macro_roc_auc": nan_safe(yt, yp, "roc"), "macro_pr_auc": nan_safe(yt, yp, "pr")}

train_loader, val_loader, test_loader = make_loader("train", shuffle=True), make_loader("validation"), make_loader("test")
best_macro_map = 0.0
ckpt = CKPT_DIR / "baseline"; ckpt.mkdir(parents=True, exist_ok=True)
hist = []
for epoch in range(1, 11):
    model.train(); total = 0
    for x, y in tqdm(train_loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
        total += loss.item() * len(x)
    vm = evaluate(val_loader)
    hist.append({"epoch": epoch, "loss": total/len(train_loader.dataset), **vm})
    print(epoch, hist[-1])
    if vm["macro_pr_auc"] > best_macro_map:
        best_macro_map = vm["macro_pr_auc"]
        torch.save({"model": model.state_dict(), "tags": TAG_NAMES, "best_macro_map": best_macro_map, "epoch": epoch}, ckpt/"best.pt")
        print("  ✓ saved", best_macro_map)

state = torch.load(ckpt/"best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model"])
test_m = evaluate(test_loader)
print("TEST split-0", test_m)
pd.DataFrame(hist).to_csv(RESULTS_DIR/"02_baseline_history.csv", index=False)
(RESULTS_DIR/"02_baseline_test.json").write_text(json.dumps(test_m, indent=2))
